# Reaction path QM: geometry optimization, barriers, normal modes, ESP/RESP/CHELPG charges

This notebook processes each `RS` / `TS` / `PS` structure from the corrected Friedel-Crafts mechanism (`step_1_1`, `step_1_2`, `step_1_3`, `step_2_1`, `step_2_2a`, `step_2_2b`) **one at a time** — there is no automatic loop over all 18 structures. Each has its own cell further down so you can run/inspect/checkpoint one before moving to the next. For every structure:

1. **Optimizes the geometry first**, using the PDB coordinates only as the starting guess:
   - `RS` and `PS` are optimized to a true **minimum** (`transition=False`).
   - `TS` is optimized to a true first-order **saddle point** (`transition=True`, with a      Hessian computed at the start of the search so geomeTRIC can follow the correct mode).
2. **Reaction barrier** ΔE‡ = E(TS_opt) − E(RS_opt) and **reaction energy** ΔE = E(PS_opt) − E(RS_opt), in kcal/mol, computed from the *optimized* energies.
3. **Normal modes / vibrational frequencies**, computed at the *optimized* geometry (this is the only way a harmonic analysis is physically meaningful — it requires a stationary point).
4. **ESP, RESP, and CHELPG atomic point charges**, computed at the *optimized* geometry.

All structures are formally neutral overall (`REMARK 15 TOTAL FORMAL CHARGE +0` in every file), singlet, so charge=0 / multiplicity=1 is used throughout.

> ⚠️ **Two things worth knowing before you run this:**
>
> - **Cost.** TS optimization needs an analytic Hessian at the starting geometry (`hessian='first+last'`), which is the most expensive single step in this notebook — for a ~60-atom system at B3LYP/def2-SVP this can take a long time on a workstation. Consider running step 1.1 alone first (see the `STEPS` list below) before committing to all six.
> - **RS/PS drift.** `RS` and `PS` here are *non-covalently bound pre-/post-reaction complexes* (held together by H-bonds/electrostatics, not a covalent bond). An **unconstrained** minimization can occasionally let such a complex drift to a different local minimum (e.g. the two fragments separating further, or a water rotating) than the specific arrangement the PDB was built to represent. Always compare `rmsd_to_input_A` (printed for every structure) against the optimized structure — a value under ~0.3–0.5 Å is typically fine; a large value means the complex relaxed into something different and you may want to add distance/dihedral constraints in geomeTRIC before trusting that barrier.


In [ ]:
# Local environment check (same pattern as your other notebooks).
# Run from the quimica-echem conda environment: conda activate quimica-echem
import os
import glob
import json
import pickle
import numpy as np
import pandas as pd
import veloxchem as vlx

required = [
    "ScfRestrictedDriver", "ScfGradientDriver", "OptimizationDriver",
    "VibrationalAnalysis", "EspChargesDriver", "RespChargesDriver",
    "MolecularBasis", "Molecule",
]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")


In [ ]:
# --- Configuration ---------------------------------------------------------

# Folder that holds the *.pdb files. Defaults to the notebook's own folder
# (the pdbs are shipped alongside it); change if you keep them elsewhere.
PDB_DIR = "."

BASIS_LABEL = "def2-svp"
FUNCTIONAL = "b3lyp"
DISPERSION = True
GRID_LEVEL = 4              # DFT integration grid: 1 (fastest/coarsest) - 7 (finest). 4 is a reasonable speed/accuracy compromise for optimization.
CONV_THRESH = 1e-4          # loose-ish SCF threshold, matches your AtomicCharges notebook
OPT_MAX_ITER = 150
HARTREE2KCAL = 627.509474

# Where intermediate results get checkpointed, so a crash/interrupt doesn't
# force you to redo everything from scratch.
CHECKPOINT_PATH = "reaction_qm_results.pkl"

# Where the per-step PDB snapshots (optimization / energy / normal modes /
# charges) get written, one set per structure.
OUTPUT_PDB_DIR = "structure_pdbs"
os.makedirs(OUTPUT_PDB_DIR, exist_ok=True)

# The six RS/TS/PS triplets, in mechanistic order. Comment out rows to run a
# subset first (e.g. just "1.1") since TS optimization is the expensive step.
STEPS = [
    ("1.1", "Nucleophilic N-C addition (pAF amine + trans-2-hexenal)"),
    ("1.2", "Two-water proton redistribution"),
    ("1.3", "Dehydration to iminium"),
    ("2.1", "Friedel-Crafts C-C bond formation (indole)"),
    ("2.2a", "W2 hydroxide deprotonates indolium"),
    ("2.2b", "W1 donates proton to alpha carbon"),
]

def step_files(step_id):
    tag = step_id.replace(".", "_")
    return {
        state: os.path.join(PDB_DIR, f"step_{tag}_{state}.pdb")
        for state in ("RS", "TS", "PS")
    }

for step_id, _ in STEPS:
    for state, path in step_files(step_id).items():
        assert os.path.exists(path), f"Missing file: {path}"
print("All PDB files found.")


In [ ]:
# --- PDB -> VeloxChem Molecule ----------------------------------------------

def parse_pdb(pdb_path):
    """Read ATOM/HETATM records from a PDB file.
    Returns (labels, coords) with coords in Angstrom, in file order.
    Element symbol is taken from columns 77-78 (standard PDB element field).
    """
    labels, coords = [], []
    with open(pdb_path) as fh:
        for line in fh:
            if line.startswith(("ATOM", "HETATM")):
                element = line[76:78].strip()
                if not element:
                    # fallback: strip digits/symbols from the atom name field
                    raw = line[12:16].strip()
                    element = "".join(c for c in raw if c.isalpha())[:2].capitalize()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                labels.append(element)
                coords.append((x, y, z))
    return labels, np.array(coords, dtype=float)


def build_molecule(labels, coords, charge=0, multiplicity=1):
    xyz_lines = [f"{lab:2s} {x: .8f} {y: .8f} {z: .8f}" for lab, (x, y, z) in zip(labels, coords)]
    xyz_str = "\n".join(xyz_lines) + "\n"
    molecule = vlx.Molecule.read_str(xyz_str)
    molecule.set_charge(charge)
    molecule.set_multiplicity(multiplicity)
    return molecule


def load_molecule(pdb_path, charge=0, multiplicity=1):
    labels, coords = parse_pdb(pdb_path)
    molecule = build_molecule(labels, coords, charge=charge, multiplicity=multiplicity)
    return molecule, labels, coords


def write_pdb(pdb_path, labels, coords=None, occupancies=None, bfactors=None,
              resname="MOL", chain="A", resseq=1, remarks=None, models=None):
    """Write a minimal-but-valid PDB file for one structure snapshot.

    - Single geometry: pass `coords` as an (n_atoms, 3) array.
    - Multi-frame trajectory (e.g. a normal-mode animation): pass `models` as a
      list of (n_atoms, 3) arrays instead; `coords` is then ignored.
    - `bfactors` (and/or `occupancies`) are optional per-atom float arrays --
      handy for dropping ESP/RESP/CHELPG charges into the B-factor column so
      they can be colored/visualized directly in a molecular viewer (PyMOL,
      VMD, py3Dmol "cartoon.colorscheme": {"prop": "b"}, etc.).
    - `remarks` is a free-text block written out as leading REMARK lines.
    """
    frames = models if models is not None else [coords]
    n_atoms = len(labels)
    occ = np.asarray(occupancies, dtype=float) if occupancies is not None else np.ones(n_atoms)
    bf = np.asarray(bfactors, dtype=float) if bfactors is not None else np.zeros(n_atoms)

    with open(pdb_path, "w") as fh:
        if remarks:
            for line in remarks.splitlines():
                fh.write(f"REMARK   {line}\n")
        multi = len(frames) > 1
        for model_idx, frame in enumerate(frames, start=1):
            if multi:
                fh.write(f"MODEL     {model_idx:4d}\n")
            for i, (label, xyz) in enumerate(zip(labels, frame)):
                x, y, z = xyz
                name = label.strip()[:4]
                fh.write(
                    f"HETATM{i + 1:5d} {name:<4s} {resname:<3s} {chain}{resseq:4d}    "
                    f"{x:8.3f}{y:8.3f}{z:8.3f}{occ[i]:6.2f}{bf[i]:6.2f}          "
                    f"{label.strip():>2s}\n"
                )
            fh.write("TER\n")
            if multi:
                fh.write("ENDMDL\n")
        fh.write("END\n")
    return pdb_path


In [ ]:
# --- Checkpointing helpers ---------------------------------------------------

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "rb") as fh:
            return pickle.load(fh)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_PATH, "wb") as fh:
        pickle.dump(results, fh)

results = load_checkpoint()
print(f"Loaded {len(results)} cached structure result(s) from {CHECKPOINT_PATH}" if results else "Starting fresh (no checkpoint yet).")


In [ ]:
# --- Geometry optimization (minimum for RS/PS, saddle point for TS) ---------
# Follows VeloxChem's documented TS-search recipe: build a ScfGradientDriver,
# wrap it in an OptimizationDriver, and set .transition/.hessian accordingly.
# Reference: https://veloxchem.org/docs/pes.html and the eChem TS tutorial.

def optimize_geometry(molecule, basis, scf_drv, transition):
    scf_grad_drv = vlx.ScfGradientDriver(scf_drv)
    if hasattr(scf_grad_drv, "ostream"):
        scf_grad_drv.ostream.mute()

    opt_drv = vlx.OptimizationDriver(scf_grad_drv)
    if hasattr(opt_drv, "ostream"):
        opt_drv.ostream.mute()
    opt_drv.transition = transition
    # Only pay for a Hessian where it's actually needed: TS search needs one
    # at the START to know which mode to follow. We deliberately skip the
    # final Hessian ('last') here -- it's the single most expensive step in
    # the whole optimization, and it's redundant anyway since Stage 3
    # (compute_normal_modes) already runs its own independent frequency
    # calculation on the optimized geometry.
    opt_drv.hessian = "first" if transition else "never"
    if hasattr(opt_drv, "max_iter"):
        opt_drv.max_iter = OPT_MAX_ITER

    opt_results = opt_drv.compute(molecule, basis, scf_drv)

    # Extract the optimized molecule. VeloxChem versions differ slightly in
    # what key/attribute carries the final structure -- try the documented
    # one first, then fall back to other plausible names, then to whatever
    # the driver/molecule object holds after compute() returns.
    opt_molecule = None
    if isinstance(opt_results, dict):
        for k in ("final_molecule", "opt_molecule", "molecule"):
            if k in opt_results:
                opt_molecule = opt_results[k]
                break
    if opt_molecule is None:
        for attr in ("final_molecule", "opt_molecule"):
            if hasattr(opt_drv, attr):
                opt_molecule = getattr(opt_drv, attr)
                break
    if opt_molecule is None:
        # geomeTRIC updates the molecule object in place in some versions
        opt_molecule = molecule
        print("    [warning] could not find an explicit optimized-molecule key/attr; "
              "falling back to the (possibly already-updated) input Molecule object. "
              f"opt_results type: {type(opt_results)}, "
              f"keys: {list(opt_results.keys()) if isinstance(opt_results, dict) else 'n/a'}")

    n_opt_steps = None
    if isinstance(opt_results, dict):
        for k in ("opt_energies", "energies"):
            if k in opt_results:
                n_opt_steps = len(opt_results[k])
                break

    return opt_molecule, opt_results, n_opt_steps


In [ ]:
# --- Four separate, individually-checkpointed stages for ONE structure -----
# Each stage reads/writes results[key] and calls save_checkpoint() itself, so
# you can run just optimize_structure(), inspect it, and only then decide to
# spend time on compute_normal_modes() etc. Each stage also writes its own
# PDB snapshot. A stage refuses to run until the previous one has completed
# for that key (checked via results[key]["stage"]).

_STAGE_ORDER = ["optimized", "energy", "vib", "charges"]


def _rebuild_scf(res):
    """Re-build the Molecule + re-run SCF at the already-optimized geometry
    stored in `res`. Needed at the start of every stage after optimization,
    since we don't keep the (unpicklable) VeloxChem driver/molecule objects
    in the checkpoint -- only plain arrays.
    """
    molecule = build_molecule(
        res["labels"], res["optimized_coords_A"],
        charge=res["charge"], multiplicity=res["multiplicity"],
    )
    basis = vlx.MolecularBasis.read(molecule, BASIS_LABEL)
    scf_drv = vlx.ScfRestrictedDriver()
    scf_drv.xcfun = FUNCTIONAL
    scf_drv.dispersion = DISPERSION
    scf_drv.grid_level = GRID_LEVEL
    scf_drv.update_settings({"conv_thresh": CONV_THRESH})
    scf_drv.ostream.mute()
    scf_results = scf_drv.compute(molecule, basis)
    return molecule, basis, scf_drv, scf_results


def optimize_structure(pdb_path, state, charge=0, multiplicity=1):
    """Stage 1/4: geometry optimization (minimum for RS/PS, saddle-point
    search for TS). Writes '{key}_1_optimized.pdb'.
    """
    pdb_path = os.path.join(PDB_DIR, pdb_path) if not os.path.isabs(pdb_path) and not os.path.exists(pdb_path) else pdb_path
    key = os.path.splitext(os.path.basename(pdb_path))[0]

    if key in results and results[key].get("stage") in _STAGE_ORDER:
        res = results[key]
        print(f"[cached] {key}: already optimized  "
              f"RMSD={res['rmsd_to_input_A']:.3f} A  opt_steps={res['n_opt_steps']}")
        return res

    print(f"[1/4 optimizing] {key}  ({pdb_path})  -> "
          f"{'TS (saddle-point search)' if state == 'TS' else 'minimum'}")

    labels, input_coords = parse_pdb(pdb_path)
    molecule = build_molecule(labels, input_coords, charge=charge, multiplicity=multiplicity)
    basis = vlx.MolecularBasis.read(molecule, BASIS_LABEL)

    # Initial SCF at the input (PDB) geometry -- required before the gradient /
    # optimization driver can take over.
    scf_drv = vlx.ScfRestrictedDriver()
    scf_drv.xcfun = FUNCTIONAL
    scf_drv.dispersion = DISPERSION
    scf_drv.grid_level = GRID_LEVEL
    scf_drv.update_settings({"conv_thresh": CONV_THRESH})
    scf_drv.ostream.mute()
    scf_drv.compute(molecule, basis)

    transition = (state == "TS")
    opt_molecule, opt_results, n_opt_steps = optimize_geometry(molecule, basis, scf_drv, transition)

    opt_labels = list(opt_molecule.get_labels())
    opt_coords = np.array(opt_molecule.get_coordinates_in_angstrom(), dtype=float)
    rmsd = float(np.sqrt(np.mean(np.sum((opt_coords - input_coords) ** 2, axis=1))))

    pdb_optimized = os.path.join(OUTPUT_PDB_DIR, f"{key}_1_optimized.pdb")
    write_pdb(
        pdb_optimized, opt_labels, opt_coords,
        remarks=(f"Step: optimization\n"
                 f"Structure: {key} ({state})\n"
                 f"RMSD to input PDB: {rmsd:.3f} A over {n_opt_steps} opt step(s)\n"
                 f"Transition-state search: {transition}"),
    )

    res = {
        "key": key,
        "pdb": pdb_path,
        "state": state,
        "labels": opt_labels,
        "charge": charge,
        "multiplicity": multiplicity,
        "input_coords_A": input_coords,
        "optimized_coords_A": opt_coords,
        "rmsd_to_input_A": rmsd,
        "n_opt_steps": n_opt_steps,
        "transition_search": transition,
        "pdb_optimized": pdb_optimized,
        "stage": "optimized",
    }
    results[key] = res
    save_checkpoint(results)

    print(f"    RMSD to input = {rmsd:.3f} A   opt steps = {n_opt_steps}")
    print(f"    PDB -> {pdb_optimized}")
    return res


def compute_energy(key):
    """Stage 2/4: QM barrier/reaction energy -- re-run SCF at the optimized
    geometry and record the energy. Writes '{key}_2_energy.pdb'.
    """
    if key not in results or results[key].get("stage") not in _STAGE_ORDER:
        raise KeyError(f"{key}: run optimize_structure() first.")
    res = results[key]
    if res.get("stage") in ("energy", "vib", "charges"):
        print(f"[cached] {key}: energy already computed  E = {res['energy_hartree']:.8f} Ha")
        return res

    print(f"[2/4 energy] {key}")
    opt_molecule, basis_opt, scf_drv_opt, scf_results = _rebuild_scf(res)

    energy = None
    for k in ("scf_energy", "total_energy", "energy"):
        if isinstance(scf_results, dict) and k in scf_results:
            energy = float(scf_results[k])
            break
    if energy is None:
        for attr in ("get_scf_energy", "scf_energy"):
            if hasattr(scf_drv_opt, attr):
                val = getattr(scf_drv_opt, attr)
                energy = float(val() if callable(val) else val)
                break
    if energy is None:
        raise RuntimeError(
            f"Could not find SCF energy for {key}. "
            f"Available keys: {list(scf_results.keys()) if isinstance(scf_results, dict) else scf_results}"
        )

    pdb_energy = os.path.join(OUTPUT_PDB_DIR, f"{key}_2_energy.pdb")
    write_pdb(
        pdb_energy, res["labels"], res["optimized_coords_A"],
        remarks=(f"Step: single-point energy\n"
                 f"Structure: {key} ({res['state']})\n"
                 f"SCF energy ({FUNCTIONAL}/{BASIS_LABEL}): {energy:.8f} Hartree"),
    )

    res["energy_hartree"] = energy
    res["pdb_energy"] = pdb_energy
    res["stage"] = "energy"
    save_checkpoint(results)

    print(f"    E_opt = {energy:.8f} Hartree")
    print(f"    PDB -> {pdb_energy}")
    return res


def compute_normal_modes(key):
    """Stage 3/4: harmonic vibrational analysis at the optimized geometry.
    Writes an animated-mode trajectory to '{key}_3_normal_mode.pdb'.
    """
    if key not in results or results[key].get("stage") not in _STAGE_ORDER:
        raise KeyError(f"{key}: run optimize_structure() first.")
    res = results[key]
    if res.get("stage") not in ("energy", "vib", "charges"):
        raise KeyError(f"{key}: run compute_energy() first.")
    if res.get("stage") in ("vib", "charges"):
        print(f"[cached] {key}: normal modes already computed  n_imaginary={res.get('n_imaginary')}")
        return res

    print(f"[3/4 normal modes] {key}")
    opt_molecule, basis_opt, scf_drv_opt, scf_results = _rebuild_scf(res)

    vib_drv = vlx.VibrationalAnalysis(scf_drv_opt)
    vib_drv.ostream.mute()
    vib_drv.do_ir = True
    vib_results = vib_drv.compute(opt_molecule, basis_opt)
    frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
    res["frequencies_cm1"] = frequencies
    res["normal_modes"] = np.array(vib_results["normal_modes"], dtype=float)
    res["reduced_masses"] = np.array(vib_results["reduced_masses"], dtype=float)
    res["force_constants"] = np.array(vib_results["force_constants"], dtype=float)
    res["ir_intensities"] = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)
    res["n_imaginary"] = int(np.sum(frequencies < 0))

    # Animate the imaginary reaction-coordinate mode (TS) or the lowest real
    # mode (RS/PS) as a multi-MODEL PDB trajectory.
    mode_idx = int(np.argmin(frequencies))
    mode_vec = res["normal_modes"][mode_idx]
    n_frames, amplitude = 20, 0.45
    mode_frames = [
        res["optimized_coords_A"] + amplitude * np.sin(phase) * mode_vec
        for phase in np.linspace(0.0, 2.0 * np.pi, n_frames, endpoint=False)
    ]
    pdb_normal_mode = os.path.join(OUTPUT_PDB_DIR, f"{key}_3_normal_mode.pdb")
    write_pdb(
        pdb_normal_mode, res["labels"], models=mode_frames,
        remarks=(f"Step: normal modes\n"
                 f"Structure: {key} ({res['state']})\n"
                 f"Animated mode index {mode_idx}: {frequencies[mode_idx]:.1f} cm-1 "
                 f"({'imaginary' if frequencies[mode_idx] < 0 else 'real'})\n"
                 f"n_imaginary (total) = {res['n_imaginary']}"),
    )
    res["pdb_normal_mode"] = pdb_normal_mode
    res["stage"] = "vib"
    save_checkpoint(results)

    n_imag = res["n_imaginary"]
    print(f"    n_imaginary = {n_imag}")
    if res["state"] == "TS" and n_imag != 1:
        print(f"    [check] expected exactly 1 imaginary frequency for a clean TS, got {n_imag}.")
    if res["state"] in ("RS", "PS") and n_imag != 0:
        print(f"    [check] expected 0 imaginary frequencies for a minimum, got {n_imag}.")
    print(f"    PDB -> {pdb_normal_mode}")
    return res


def compute_charges(key):
    """Stage 4/4: ESP / RESP / CHELPG charges at the optimized geometry.
    Writes one PDB per charge scheme, each with charges in the B-factor column.
    """
    if key not in results or results[key].get("stage") not in _STAGE_ORDER:
        raise KeyError(f"{key}: run optimize_structure() first.")
    res = results[key]
    if res.get("stage") != "vib" and res.get("stage") != "charges":
        raise KeyError(f"{key}: run compute_normal_modes() first.")
    if res.get("stage") == "charges":
        print(f"[cached] {key}: charges already computed")
        return res

    print(f"[4/4 charges] {key}")
    opt_molecule, basis_opt, scf_drv_opt, scf_results = _rebuild_scf(res)

    esp_drv = vlx.EspChargesDriver()
    esp_drv.ostream.mute()
    esp_charges = esp_drv.compute(opt_molecule, basis_opt, scf_results)

    resp_drv = vlx.RespChargesDriver()
    resp_drv.ostream.mute()
    resp_charges = resp_drv.compute(opt_molecule, basis_opt, scf_results)

    chelpg_drv = vlx.EspChargesDriver()
    chelpg_drv.ostream.mute()
    chelpg_drv.grid_type = "chelpg"
    chelpg_charges = chelpg_drv.compute(opt_molecule, basis_opt, scf_results)

    res["esp_charges"] = np.array(esp_charges, dtype=float)
    res["resp_charges"] = np.array(resp_charges, dtype=float)
    res["chelpg_charges"] = np.array(chelpg_charges, dtype=float)

    pdb_esp = os.path.join(OUTPUT_PDB_DIR, f"{key}_4_esp_charges.pdb")
    write_pdb(
        pdb_esp, res["labels"], res["optimized_coords_A"], bfactors=res["esp_charges"],
        remarks=(f"Step: charges (ESP)\nStructure: {key} ({res['state']})\n"
                 f"Per-atom ESP charge (e) written to the B-factor column."),
    )
    pdb_resp = os.path.join(OUTPUT_PDB_DIR, f"{key}_4_resp_charges.pdb")
    write_pdb(
        pdb_resp, res["labels"], res["optimized_coords_A"], bfactors=res["resp_charges"],
        remarks=(f"Step: charges (RESP)\nStructure: {key} ({res['state']})\n"
                 f"Per-atom RESP charge (e) written to the B-factor column."),
    )
    pdb_chelpg = os.path.join(OUTPUT_PDB_DIR, f"{key}_4_chelpg_charges.pdb")
    write_pdb(
        pdb_chelpg, res["labels"], res["optimized_coords_A"], bfactors=res["chelpg_charges"],
        remarks=(f"Step: charges (CHELPG)\nStructure: {key} ({res['state']})\n"
                 f"Per-atom CHELPG charge (e) written to the B-factor column."),
    )
    res["pdb_esp_charges"] = pdb_esp
    res["pdb_resp_charges"] = pdb_resp
    res["pdb_chelpg_charges"] = pdb_chelpg
    res["stage"] = "charges"
    save_checkpoint(results)

    print(f"    PDBs -> {pdb_esp}, {pdb_resp}, {pdb_chelpg}")
    return res


## Run structures one at a time — one stage at a time

No automatic loop over all 18 structures, and no single call that hides all the work either. Each cell below runs **one structure** through its four stages, one call per line:

1. `optimize_structure(pdb, state)` — minimum for RS/PS, saddle-point search for TS
2. `compute_energy(key)` — SCF energy at the optimized geometry
3. `compute_normal_modes(key)` — harmonic frequencies + normal modes
4. `compute_charges(key)` — ESP / RESP / CHELPG charges

Each stage is checkpointed to disk on its own and writes its own PDB snapshot to `structure_pdbs/`. If a run gets interrupted partway (e.g. mid vibrational-analysis), the earlier stages are already saved — re-running the cell just picks up where it left off instead of redoing everything. You can also comment out the later lines in a cell if you only want to run/inspect the optimization for now and come back for the rest later.


### Step 1.1 — Nucleophilic N-C addition (pAF amine + trans-2-hexenal)


In [ ]:
# Nucleophilic N-C addition (pAF amine + trans-2-hexenal)
res = optimize_structure("step_1_1_RS.pdb", "RS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Nucleophilic N-C addition (pAF amine + trans-2-hexenal)
res = optimize_structure("step_1_1_TS.pdb", "TS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Nucleophilic N-C addition (pAF amine + trans-2-hexenal)
res = optimize_structure("step_1_1_PS.pdb", "PS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


### Step 1.2 — Two-water proton redistribution


In [ ]:
# Two-water proton redistribution
res = optimize_structure("step_1_2_RS.pdb", "RS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Two-water proton redistribution
res = optimize_structure("step_1_2_TS.pdb", "TS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Two-water proton redistribution
res = optimize_structure("step_1_2_PS.pdb", "PS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


### Step 1.3 — Dehydration to iminium


In [ ]:
# Dehydration to iminium
res = optimize_structure("step_1_3_RS.pdb", "RS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Dehydration to iminium
res = optimize_structure("step_1_3_TS.pdb", "TS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Dehydration to iminium
res = optimize_structure("step_1_3_PS.pdb", "PS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


### Step 2.1 — Friedel-Crafts C-C bond formation (indole)


In [ ]:
# Friedel-Crafts C-C bond formation (indole)
res = optimize_structure("step_2_1_RS.pdb", "RS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Friedel-Crafts C-C bond formation (indole)
res = optimize_structure("step_2_1_TS.pdb", "TS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# Friedel-Crafts C-C bond formation (indole)
res = optimize_structure("step_2_1_PS.pdb", "PS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


### Step 2.2a — W2 hydroxide deprotonates indolium


In [ ]:
# W2 hydroxide deprotonates indolium
res = optimize_structure("step_2_2a_RS.pdb", "RS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# W2 hydroxide deprotonates indolium
res = optimize_structure("step_2_2a_TS.pdb", "TS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# W2 hydroxide deprotonates indolium
res = optimize_structure("step_2_2a_PS.pdb", "PS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


### Step 2.2b — W1 donates proton to alpha carbon


In [ ]:
# W1 donates proton to alpha carbon
res = optimize_structure("step_2_2b_RS.pdb", "RS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# W1 donates proton to alpha carbon
res = optimize_structure("step_2_2b_TS.pdb", "TS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


In [ ]:
# W1 donates proton to alpha carbon
res = optimize_structure("step_2_2b_PS.pdb", "PS")
res = compute_energy(res["key"])
res = compute_normal_modes(res["key"])
res = compute_charges(res["key"])


## Barriers and reaction energies (from optimized geometries)

This and every cell below only uses whatever is currently in `results` — run it any time after finishing at least one full RS/TS/PS triplet above; steps you haven't run yet are simply skipped (with a printed note).


In [ ]:
# --- Barrier / reaction-energy summary table --------------------------------

rows = []
for step_id, description in STEPS:
    tag = step_id.replace(".", "_")
    keys = [f"step_{tag}_{s}" for s in ("RS", "TS", "PS")]
    if not all(k in results for k in keys):
        missing = [k for k in keys if k not in results]
        print(f"[skip] step {step_id}: not run yet ({', '.join(missing)})")
        continue
    rs, ts, ps = (results[k] for k in keys)
    barrier = (ts["energy_hartree"] - rs["energy_hartree"]) * HARTREE2KCAL
    rxn_energy = (ps["energy_hartree"] - rs["energy_hartree"]) * HARTREE2KCAL
    rows.append({
        "step": step_id,
        "description": description,
        "E_RS_opt (Ha)": rs["energy_hartree"],
        "E_TS_opt (Ha)": ts["energy_hartree"],
        "E_PS_opt (Ha)": ps["energy_hartree"],
        "barrier ddE‡ (kcal/mol)": barrier,
        "reaction ddE (kcal/mol)": rxn_energy,
        "TS n_imaginary_freq": ts.get("n_imaginary"),
        "RS rmsd_to_input (A)": rs["rmsd_to_input_A"],
        "TS rmsd_to_input (A)": ts["rmsd_to_input_A"],
        "PS rmsd_to_input (A)": ps["rmsd_to_input_A"],
    })

summary = pd.DataFrame(rows)
summary.to_csv("barrier_summary.csv", index=False)
summary


## Normal modes (at the optimized geometry)

For each structure, `results[key]["frequencies_cm1"]` holds the harmonic frequencies and `results[key]["normal_modes"]` the corresponding displacement vectors, both computed at the *optimized* geometry. For every TS, `n_imaginary` should be exactly 1 (the reaction-coordinate mode); inspect its displacement pattern below to confirm it matches the bond being formed/broken in that step. For every RS/PS, `n_imaginary` should be 0 -- if it isn't, that minimization didn't fully converge to a true minimum and should be rerun (e.g. with a tighter `conv_thresh` or more `OPT_MAX_ITER` steps).


In [ ]:
# --- Frequency table (lowest 6 modes of every structure, to spot imaginary modes) ---

freq_rows = []
for step_id, _ in STEPS:
    tag = step_id.replace(".", "_")
    for state in ("RS", "TS", "PS"):
        key = f"step_{tag}_{state}"
        if key not in results:
            continue
        freqs = results[key]["frequencies_cm1"]
        lowest6 = np.sort(freqs)[:6]
        freq_rows.append({
            "key": key,
            "n_imaginary": results[key]["n_imaginary"],
            "lowest_6_freqs_cm-1": np.round(lowest6, 1).tolist(),
        })

freq_table = pd.DataFrame(freq_rows)
freq_table.to_csv("frequency_summary.csv", index=False)
freq_table


In [ ]:
# --- Inspect / animate a specific normal mode (e.g. the TS reaction coordinate) -----
import py3Dmol

def normal_mode_trajectory(labels, coords, mode, amplitude=0.45, frames=32):
    mode = np.array(mode, dtype=float)
    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(key, mode_index=0, amplitude=0.45, frames=32):
    res = results[key]
    # NOTE: uses the OPTIMIZED coordinates, not the raw PDB input.
    traj = normal_mode_trajectory(res["labels"], res["optimized_coords_A"], res["normal_modes"][mode_index], amplitude, frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Example: the imaginary (reaction-coordinate) mode of the step 1.1 TS is
# whichever index has a negative frequency -- usually mode 0.
example_key = "step_1_1_TS"
neg_idx = int(np.argmin(results[example_key]["frequencies_cm1"]))
print(f"{example_key}: lowest mode index {neg_idx}, freq = {results[example_key]['frequencies_cm1'][neg_idx]:.1f} cm^-1")
show_mode(example_key, mode_index=neg_idx).show()


## ESP / RESP / CHELPG charges (at the optimized geometry)


In [ ]:
# --- Export atomic charges for every structure to one tidy CSV --------------

charge_rows = []
for step_id, _ in STEPS:
    tag = step_id.replace(".", "_")
    for state in ("RS", "TS", "PS"):
        key = f"step_{tag}_{state}"
        if key not in results:
            continue
        res = results[key]
        for i, label in enumerate(res["labels"]):
            charge_rows.append({
                "key": key,
                "atom_index": i + 1,
                "element": label,
                "esp_charge": res["esp_charges"][i],
                "resp_charge": res["resp_charges"][i],
                "chelpg_charge": res["chelpg_charges"][i],
            })

charges_table = pd.DataFrame(charge_rows)
charges_table.to_csv("atomic_charges_all_structures.csv", index=False)
charges_table.head(20)


## Notes

- **Everything downstream (barriers, normal modes, charges) is now computed from the optimized geometry, not the raw PDB coordinates.** The raw PDB is only the starting guess fed to `OptimizationDriver`.
- `RS`/`PS` → energy-minimized (`transition=False`). `TS` → saddle-point search (`transition=True`, Hessian at the first and last step) following VeloxChem's documented TS procedure (geomeTRIC under the hood).
- `rmsd_to_input_A` in `barrier_summary.csv` tells you how far each structure moved from the PDB starting guess during optimization — check this, especially for RS/PS, to make sure the optimizer didn't relax into a qualitatively different arrangement (see the warning at the top of the notebook).
- `barrier_summary.csv` — ΔE‡ and ΔE_rxn (kcal/mol, optimized-geometry energies) for all 6 steps, plus TS imaginary-frequency count and RMSD-to-input for every state.
- `frequency_summary.csv` — lowest 6 harmonic frequencies for every one of the 18 optimized structures.
- `atomic_charges_all_structures.csv` — ESP, RESP, and CHELPG charges per atom, computed post-optimization, for every structure.
- `reaction_qm_results.pkl` — full checkpoint (optimized coordinates, energies, all frequencies/normal modes, all charges) so you can re-open this notebook and continue analysis without recomputation.
- If a TS doesn't converge to exactly one imaginary frequency, or `n_opt_steps` hits `OPT_MAX_ITER` without converging, that structure's starting guess likely needs to be closer to the true saddle point (e.g. tighten the forming/breaking bond distance in the PDB) before re-running.
